In [8]:
import pandas as pd
import json

DATA_PATH = "C:\\Users\\USER\\Desktop\\MediRAG\\data division logic\\miriad_neurology.jsonl"

# ── 1. Load ────────────────────────────────────────────────────────────────
df = pd.read_json(DATA_PATH, lines=True)   # change to read_csv if needed

# ── 2. Inspect ─────────────────────────────────────────────────────────────
print(df.shape)
print(df.columns.tolist())
print(df["specialty"].value_counts())
print(df.head(2))

(205154, 11)
['qa_id', 'paper_id', 'question', 'answer', 'paper_url', 'paper_title', 'passage_text', 'passage_position', 'year', 'venue', 'specialty']
specialty
Neurology       198125
Neurosurgery      7029
Name: count, dtype: int64
          qa_id  paper_id                                           question  \
0  381778036701  17780367  What are the mechanisms underlying stroke in c...   
1   38833531102   8335311  How is the volume of intracerebral hemorrhage ...   

                                              answer  \
0  The mechanisms underlying stroke in cancer pat...   
1  The volume of intracerebral hemorrhage is meas...   

                                           paper_url  \
0  https://api.semanticscholar.org/CorpusID:17780367   
1   https://api.semanticscholar.org/CorpusID:8335311   

                                         paper_title  \
0  Clues to Occult Cancer in Patients with Ischem...   
1  Hemorrhage Prediction of 30-Day Mortality Amon...   

                   

In [9]:
import pandas as pd

# ── Filter by specialty ────────────────────────────────────────────────────
neuro     = df[df["specialty"] == "Neurology"]
neurosurg = df[df["specialty"] == "Neurosurgery"]

# ── Sample ─────────────────────────────────────────────────────────────────
neuro_sample     = neuro.sample(n=1750, random_state=42)
neurosurg_sample = neurosurg.sample(n=250, random_state=42)

sampled = pd.concat([neuro_sample, neurosurg_sample]).reset_index(drop=True)

# ── Keep only what we need ─────────────────────────────────────────────────
sampled = sampled[["question", "answer", "specialty"]]

print(f"Total sampled : {len(sampled)}")
print(sampled["specialty"].value_counts())
print(sampled.head(2))

# ── Save ───────────────────────────────────────────────────────────────────
sampled.to_json("sampled_2k.jsonl", orient="records", lines=True)
print("Saved to sampled_3k.jsonl")

Total sampled : 2000
specialty
Neurology       1750
Neurosurgery     250
Name: count, dtype: int64
                                            question  \
0  How does transcranial magnetic stimulation (TM...   
1  How does miR-124-3p play a role in nervous sys...   

                                              answer  specialty  
0  Transcranial magnetic stimulation (TMS) is a t...  Neurology  
1  miR-124-3p is involved in nervous system disea...  Neurology  
Saved to sampled_3k.jsonl


In [4]:
from openai import OpenAI
import json
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found")

client = OpenAI(api_key=api_key)

print("OpenAI client ready for GPT-4o0mini calls.")

# ── Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a senior Neurologist and Neurosurgeon.

Your task is to generate an ordered set of answer requirements that will guide relevance reranking of retrieved passages.

All questions belong to Neurology and Neurosurgery.

STRICT CONSTRAINTS:

- Use ONLY the clinical question to derive requirements.
- Each requirement must be directly inferable from the grammatical structure of the question.
- Do NOT expand beyond what the question explicitly asks.
- Do NOT introduce safety, contraindications, clinical trials, epidemiology, prognosis, or limitations unless explicitly mentioned in the question.
- Do NOT create a comprehensive textbook checklist.
- Focus only on the core semantic dimensions required to answer THIS question.

STRUCTURE:

- Generate 3–4 ordered requirements.
- Order them by importance (most important first).
- Requirements must be concise and discriminative.
- Avoid vague phrasing.
- Avoid adding evaluation language (e.g., "rank higher if").

Return ONLY valid JSON.

Output format:
{"blueprint": ["1. ...", "2. ...", "3. ..."]}
"""


def build_user_prompt(question: str) -> str:
    return f"Clinical Question: {question}"


def call_blueprint(question: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(question)}
        ],
        temperature=0.2,        # low temp for consistent structured output
        max_tokens=300,
        response_format={"type": "json_object"}  # forces valid JSON output
    )
    raw = response.choices[0].message.content
    return json.loads(raw)


# ── Single test call ───────────────────────────────────────────────────────
import pandas as pd

sampled = pd.read_json("sampled_2k.jsonl", lines=True)

# Test on first 3 samples
for i in range(3):
    test_row = sampled.iloc[i]
    
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    
    print("QUESTION:")
    print(test_row["question"])
    print()
    
    print("ORIGINAL ANSWER:")
    print(test_row["answer"])
    print(f"\nWord count: {len(test_row['answer'].split())}")
    print()
    
    result = call_blueprint(test_row["question"])
    
    print("GENERATED BLUEPRINT:")
    print(json.dumps(result, indent=2))


# # ── Save first 20 blueprints ─────────────────────────────────────────────────
# blueprint_records = []
# limit = min(20, len(sampled))

# for i in range(limit):
#     row = sampled.iloc[i]
#     bp = call_blueprint(row["question"])
#     blueprint_records.append({
#         "index": int(i),
#         "question": row["question"],
#         "answer": row["answer"],
#         "specialty": row["specialty"],
#         "blueprint": bp.get("blueprint", bp)
#     })

# output_path = "first20_blueprints.jsonl"
# with open(output_path, "w", encoding="utf-8") as f:
#     for record in blueprint_records:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

# print(f"Saved first {limit} blueprints to {output_path}")



OpenAI client ready for GPT-4o0mini calls.
Saved first 20 blueprints to first20_blueprints.jsonl


In [5]:
from openai import OpenAI
import json
import pandas as pd

# ── Blueprint–Answer Alignment Validator ───────────────────────────────────

VALIDATION_SYSTEM_PROMPT = """You are a medical QA evaluator specializing in Neurology and Neurosurgery.

Your task is to evaluate whether a blueprint (set of requirements) is well-aligned with a provided gold-standard answer.

For each requirement:
- Mark it as:
  "fully_satisfied" → clearly addressed in the answer
  "partially_satisfied" → somewhat addressed but incomplete
  "not_satisfied" → not meaningfully addressed

IMPORTANT:
- Judge ONLY based on the provided answer.
- Do NOT assume missing content.
- Do NOT expand interpretation.
- Be strict but fair.

Return ONLY valid JSON.

Output format:
{
  "fully_satisfied": <int>,
  "partially_satisfied": <int>,
  "not_satisfied": <int>,
  "passed": <true/false>,
  "reason": "<short explanation if failed, 'Passed' if passed>"
}

Passing criteria:
- fully_satisfied >= 2
- not_satisfied <= 1
"""


def validate_blueprint_against_answer(question: str, blueprint: list, answer: str) -> dict:
    """Validates whether blueprint aligns with gold answer."""
    
    # Basic structural checks (no API cost)
    if not isinstance(blueprint, list):
        return {"passed": False, "reason": "Blueprint is not a list"}
    
    if not (3 <= len(blueprint) <= 5):
        return {"passed": False, "reason": f"Requirement count out of range: {len(blueprint)}"}

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": VALIDATION_SYSTEM_PROMPT},
            {"role": "user", "content":
                f"Clinical Question:\n{question}\n\n"
                f"Gold Answer:\n{answer}\n\n"
                f"Blueprint Requirements:\n{json.dumps(blueprint, indent=2)}"
            }
        ],
        temperature=0.0,
        max_tokens=200,
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)


# ── Test on first 5 samples ─────────────────────────────────────────────────

sampled = pd.read_json("first20_blueprints.jsonl", lines=True)

for i in range(5):
    test_row = sampled.iloc[i]
    
    print(f"\n{'='*80}")
    print(f"ALIGNMENT VALIDATION - SAMPLE {i+1}")
    print(f"{'='*80}")
    
    print("QUESTION:")
    print(test_row["question"])
    print()
    
    print("GOLD ANSWER:")
    print(test_row["answer"])
    print()
    
    # Generate blueprint
    blueprint_result = call_blueprint(test_row["question"])
    blueprint = blueprint_result.get("blueprint", [])
    
    print("GENERATED BLUEPRINT:")
    print(json.dumps(blueprint, indent=2))
    print()
    
    # Validate alignment
    validation_result = validate_blueprint_against_answer(
        test_row["question"],
        blueprint,
        test_row["answer"]
    )
    
    print("ALIGNMENT RESULT:")
    print(json.dumps(validation_result, indent=2))


ALIGNMENT VALIDATION - SAMPLE 1
QUESTION:
How does transcranial magnetic stimulation (TMS) potentially impact epilepsy and other CNS disorders?


GOLD ANSWER:
Transcranial magnetic stimulation (TMS) is a technology that noninvasively interferes with neural activity. Low frequency repetitive TMS has shown promise in temporarily improving intractable epilepsy. While it is difficult to predict the advancement of TMS in relation to other techniques involving the chronic implantation of electrodes, it is important to consider the potential effects of repetitive stimulation on the genome. Further research is needed to explore the long-term effects of electrical stimulation on cell physiology.

GENERATED BLUEPRINT:
[
  "1. Describe the mechanism of action of transcranial magnetic stimulation (TMS) in relation to epilepsy.",
  "2. Explain the effects of TMS on seizure frequency or severity in epilepsy.",
  "3. Discuss the potential benefits of TMS for other central nervous system (CNS) disord

In [13]:
import time
import json
import pandas as pd
from tqdm import tqdm


# ── Config ─────────────────────────────────────────────────────────────────
INPUT_FILE       = "sampled_2k.jsonl"
OUTPUT_FILE      = "blueprints_2k.jsonl"
FAILED_FILE      = "blueprints_failed.jsonl"
MAX_RETRIES      = 5
RETRY_DELAY      = 3
TEMPERATURE      = 0.2


# ── Load already processed rows (safe resume) ──────────────────────────────
def load_processed_ids(output_file: str) -> set:
    processed = set()
    try:
        with open(output_file, "r") as f:
            for line in f:
                row = json.loads(line)
                processed.add(row["question"])
    except FileNotFoundError:
        pass
    return processed


# ── Generate + Validate with Retry ─────────────────────────────────────────
def process_row(question: str, answer: str, specialty: str) -> dict | None:

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # Stage 1: Generate blueprint
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(question)}
                ],
                temperature=TEMPERATURE,
                max_tokens=300,
                response_format={"type": "json_object"}
            )

            blueprint_raw = json.loads(response.choices[0].message.content)
            blueprint = blueprint_raw.get("blueprint", [])

            # Stage 2: Alignment validation against gold answer
            validation = validate_blueprint_against_answer(
                question,
                blueprint,
                answer
            )

            if validation.get("passed"):
                return {
                    "question"  : question,
                    "answer"    : answer,
                    "specialty" : specialty,
                    "blueprint" : blueprint,
                    "validation": validation
                }

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

        except Exception as e:
            print(f"[Attempt {attempt}] API error: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    return None


# ── Main Pipeline ──────────────────────────────────────────────────────────
def run_pipeline():
    sampled       = pd.read_json(INPUT_FILE, lines=True)
    processed_ids = load_processed_ids(OUTPUT_FILE)

    rows_to_process = sampled[~sampled["question"].isin(processed_ids)].reset_index(drop=True)
    total = len(sampled)
    remaining = len(rows_to_process)
    success_count = len(processed_ids)
    failed_count = 0

    print(f"Total rows   : {total}")
    print(f"Already done : {success_count}")
    print(f"Remaining    : {remaining}")
    print("─" * 60)

    if remaining == 0:
        print("Nothing left to process. Exiting.")
        return

    with open(OUTPUT_FILE, "a") as out_f, open(FAILED_FILE, "a") as fail_f:
        for _, row in tqdm(rows_to_process.iterrows(), total=remaining, desc="Processing"):
            question  = row["question"]
            answer    = row["answer"]
            specialty = row["specialty"]

            if question in processed_ids:
                continue

            result = process_row(question, answer, specialty)

            if result:
                out_f.write(json.dumps(result) + "\n")
                out_f.flush()
                success_count += 1
                processed_ids.add(question)
            else:
                fail_f.write(json.dumps({
                    "question" : question,
                    "specialty": specialty,
                    "reason"   : "Failed alignment validation"
                }) + "\n")
                fail_f.flush()
                failed_count += 1

    print("─" * 60)
    print(f"Done. Success: {success_count} | Failed: {failed_count}")
    print(f"Output saved to : {OUTPUT_FILE}")
    print(f"Failed rows     : {FAILED_FILE}")


# ── Run ────────────────────────────────────────────────────────────────────
run_pipeline()

Total rows   : 2000
Already done : 0
Remaining    : 2000
────────────────────────────────────────────────────────────


Processing: 100%|██████████| 2000/2000 [5:33:22<00:00, 10.00s/it]  

────────────────────────────────────────────────────────────
Done. Success: 1610 | Failed: 390
Output saved to : blueprints_2k.jsonl
Failed rows     : blueprints_failed.jsonl


In [3]:
import json
import pandas as pd
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────
INPUT_FILE  = "blueprints_2k.jsonl"   # your 1610 validated rows
OUTPUT_FILE = "slm2_llama3_blueprint_finetune.jsonl"


# ── Load Data ──────────────────────────────────────────────────────────────
df = pd.read_json(INPUT_FILE, lines=True)


# ── System Prompt Template ─────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a senior Neurologist and Neurosurgeon.

Your task is to generate an ordered blueprint describing what a strong answer to a clinical question must contain.

Strict rules:
- Use only the information implied by the question.
- Do not expand beyond what the question explicitly asks.
- Do not introduce unrelated subtopics.
- Generate 3 to 4 ordered requirements.
- Order them by importance (most important first).
- Keep requirements concise and clinically meaningful.
- Output must be valid JSON in the specified format.
"""


# ── Build Fine-Tuning Dataset ─────────────────────────────────────────────
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for _, row in tqdm(df.iterrows(), total=len(df)):

        question  = row["question"].strip()
        specialty = row["specialty"]
        blueprint = row["blueprint"]

        # Build system message
        system_message = SYSTEM_PROMPT

        # Assistant output (STRICT JSON string)
        assistant_content = json.dumps(
            {
                "blueprint": blueprint
            },
            ensure_ascii=False,
            indent=2
        )

        sample = {
            "conversations": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": question},
                {"role": "assistant", "content": assistant_content}
            ]
        }

        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Fine-tuning dataset created successfully.")
print(f"Saved to: {OUTPUT_FILE}")

100%|██████████| 1610/1610 [00:00<00:00, 6608.31it/s]

Fine-tuning dataset created successfully.
Saved to: slm2_llama3_blueprint_finetune.jsonl
